In [6]:
import json
import os
import requests
import pandas as pd

# 1. Setup 2026 Directory Structure
for folder in ['raw', 'processed', 'embeddings', 'models']:
    os.makedirs(f'data/{folder}', exist_ok=True)

print("📂 Directories ready.")

# 2. Download TCGA-BRCA Clinical Data (Survival targets)
def download_clinical_data():
    print("Survival data downloading...")
    url = "https://api.gdc.cancer.gov/cases"
    # Filtering for Breast Cancer (BRCA) with follow-up data [cite: 1379]
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "project.project_id", "value": ["TCGA-BRCA"]}},
            {"op": "exists", "content": {"field": "diagnoses.days_to_last_follow_up"}}
        ]
    }
    # GDC expects filters as a JSON string, not a nested dict in the query string.
    params = {
        "filters": json.dumps(filters),
        "fields": "case_id,submitter_id,diagnoses.days_to_death,diagnoses.days_to_last_follow_up,diagnoses.vital_status,diagnoses.tumor_stage",
        "format": "JSON",
        "size": "1200",
    }
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    payload = response.json()
    if "data" not in payload:
        raise RuntimeError(f"GDC API error: {payload.get('message', payload)}")
    data = payload["data"]["hits"]
    df = pd.json_normalize(data)
    df.to_csv("data/raw/clinical_brca.csv", index=False)
    print(f"✅ Downloaded clinical data for {len(df)} patients.")

download_clinical_data()

📂 Directories ready.
Survival data downloading...
✅ Downloaded clinical data for 1098 patients.


In [8]:
def get_expression_manifest():
    print("🔍 Searching for mRNA-seq files for your clinical cohort...")
    clinical_df = pd.read_csv("data/raw/clinical_brca.csv")
    case_ids = clinical_df["case_id"].tolist()

    files_url = "https://api.gdc.cancer.gov/files"
    # Use POST: ~1000 case UUIDs in a GET query string exceeds limits and can drop the connection.
    # Use op "in" with one-element lists for data_type / workflow — GDC often ignores "==" on files.
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.case_id", "value": case_ids}},
            {"op": "in", "content": {"field": "data_type", "value": ["Gene Expression Quantification"]}},
            {"op": "in", "content": {"field": "analysis.workflow_type", "value": ["STAR - Counts"]}},
        ],
    }

    page_size = 5000
    offset = 0
    hits = []
    total = 0
    while True:
        body = {
            "filters": filters,
            "fields": "file_id,file_name,file_size,md5sum,cases.case_id",
            "format": "JSON",
            "size": page_size,
            "from": offset,
        }
        response = requests.post(
            files_url,
            json=body,
            headers={"Content-Type": "application/json"},
            timeout=300,
        )
        response.raise_for_status()
        payload = response.json()
        if "data" not in payload:
            raise RuntimeError(f"GDC API error: {payload.get('message', payload)}")
        total = payload["data"]["pagination"]["total"]
        batch = payload["data"]["hits"]
        hits.extend(batch)
        offset += len(batch)
        if offset >= total or not batch:
            break

    df = pd.json_normalize(hits)
    out_path = "data/raw/gdc_manifest.txt"
    df.to_csv(out_path, sep="\t", index=False)
    print(f"✅ Manifest created: {out_path} ({len(df)} rows, {total} total matches).")
    print("Next step: download with the GDC Data Transfer Tool (on the order of a few GB).")


get_expression_manifest()

🔍 Searching for mRNA-seq files for your clinical cohort...
✅ Manifest created: data/raw/gdc_manifest.txt (1231 rows, 1231 total matches).
Next step: download with the GDC Data Transfer Tool (on the order of a few GB).
